# 🧪 專案名稱：這本筆記本的用途

一句話說明要做什麼。**（例：只是試驗，不改專案程式。）**

| 項目 | 內容 |
|---|---|
| 🖥️ 執行階段 | **L4 GPU**（推薦，最省點數）／**A100 或 G4 GPU**（極速，耗點較多）<br>（上方選單「執行階段 → 變更執行階段類型」；成片畫質相同，僅耗時差異） |
| ⏱️ 預估時間 | 安裝約 N 分鐘，執行約 N 分鐘 |
| 📥 要準備 | 本機 `路徑/` 裡的 `檔案` |
| 📤 會得到 | `result.zip`，放回本機 `路徑/` |

### 🚀 使用方式
1. 確認執行階段是 **L4 GPU**（若追求速度可選 **A100** 或 **G4**）。
2. 視需要調整下方「⚙️ 設定」。
3. 選單「執行階段 → **全部執行**」。
4. 跑到「步驟 3」會出現 **選擇檔案** 按鈕，上傳輸入檔。
5. 最後一步會自動下載結果 zip。

> ⚠️ 注意事項（例：為什麼要另建 Python 環境、跑過舊版要先「中斷連線並刪除執行階段」）。

## ⚙️ 設定

In [ ]:
# @title ⚙️ 設定 { display-mode: "form" }
# @markdown 說明每個參數的影響（例：越高越清楚、越慢）：
OPTION_A = True  # @param {type:"boolean"}
LEVEL = "200"  # @param ["150", "200", "300"]
print("設定：", "OPTION_A =", OPTION_A, "｜LEVEL =", LEVEL)

## ▶️ 執行

In [ ]:
# @title 🧰 準備：載入顯示工具 { display-mode: "form" }
# @markdown 只定義後面用到的顯示函式（✅ ⚠️ ❌ 卡片、表格），不安裝任何東西。
import html
import os
import subprocess
import time

from IPython.display import HTML, display

_KIND = {"ok": ("✅", "#16a34a"), "warn": ("⚠️", "#d97706"),
         "fail": ("❌", "#dc2626"), "info": ("ℹ️", "#2563eb")}


def card(kind, title, body=""):
    """顯示一張彩色提示卡片；title 會跳脫，body 可放 HTML。"""
    icon, color = _KIND[kind]
    extra = f'<div style="margin-top:4px;opacity:.85">{body}</div>' if body else ""
    display(HTML(
        f'<div style="border-left:4px solid {color};background:{color}1f;padding:8px 12px;'
        f'margin:6px 0;border-radius:6px;font-size:14px;line-height:1.5">'
        f'<b>{icon} {html.escape(title)}</b>{extra}</div>'))


def ok(title, body=""): card("ok", title, body)
def warn(title, body=""): card("warn", title, body)
def info(title, body=""): card("info", title, body)
def fail(title, body=""): card("fail", title, body)


def table_html(headers, rows):
    """產生簡單表格的 HTML（深色、淺色主題都看得清楚）。"""
    th = "".join(f'<th style="text-align:left;padding:4px 10px;border-bottom:2px solid #8884">{html.escape(str(h))}</th>' for h in headers)
    trs = "".join("<tr>" + "".join(f'<td style="padding:4px 10px;border-bottom:1px solid #8883">{c}</td>' for c in r) + "</tr>" for r in rows)
    return f'<table style="border-collapse:collapse;margin:6px 0;font-size:14px"><tr>{th}</tr>{trs}</table>'


def logbox(text, summary="詳細訊息"):
    """把長訊息收進可展開的區塊。"""
    return (f'<details><summary style="cursor:pointer">{html.escape(summary)}</summary>'
            f'<pre style="white-space:pre-wrap;font-size:12px;max-height:300px;overflow:auto">{html.escape(text)}</pre></details>')


class StopRun(Exception):
    """停止「全部執行」，但不顯示一大段 Python 錯誤追蹤。"""
    def _render_traceback_(self):
        return []


def sh(cmd, title):
    """執行指令並隱藏雜訊：成功顯示 ✅＋秒數，失敗顯示 ❌＋最後 40 行訊息並停止。"""
    t0 = time.time()
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    sec = time.time() - t0
    out = p.stdout + p.stderr
    if p.returncode != 0:
        fail(f"{title}：失敗", logbox(chr(10).join(out.splitlines()[-40:]), "最後 40 行訊息（點開看）"))
        raise StopRun(title)
    ok(f"{title}（{sec:.0f} 秒）")
    return p.stdout


ok("顯示工具已載入")

In [ ]:
# @title 1️⃣ 確認有分到 GPU { display-mode: "form" }
p = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
if p.returncode != 0:
    fail("沒有分到 GPU",
         "請到「執行階段 → 變更執行階段類型」選 <b>L4 GPU</b>（推薦省點數）、<b>A100</b> 或 <b>G4</b>（極速）→ 儲存，再重新「全部執行」。")
    raise StopRun("no gpu")
gpu_name, gpu_mem = [s.strip() for s in p.stdout.strip().splitlines()[0].split(",")]
mem_gb = int(gpu_mem) / 1024
tip = "（最省點數，畫質同旗艦）" if "L4" in gpu_name else "（極速推論，產出更快）" if any(k in gpu_name for k in ["A100", "G4", "PRO 6000"]) else ""
ok(f"GPU：{gpu_name}（顯示記憶體 {mem_gb:.1f} GB）{tip}")


In [ ]:
# @title 2️⃣ 建立獨立的 Python 環境並安裝套件 { display-mode: "form" }
# @markdown 版本與本機 `.venv` 相同（在下方 PACKAGES 鎖定版本）。已裝過會自動略過安裝。
import json
import os

PY_VERSION = "3.11"
PACKAGES = "套件A==1.0.0 套件B==2.0.0"  # 改成專案 requirements.txt 的版本
VENV_PY = "/content/venv/bin/python"
CHECK = ("import json, sys; "
         "print('@@VER' + json.dumps({'Python': sys.version.split()[0]}))")  # 加上主要套件與 GPU 檢查

if os.path.exists(VENV_PY):
    info("已有獨立環境，略過安裝")
else:
    sh("pip install -q uv", "安裝 uv")
    sh(f"uv venv -q --python {PY_VERSION} /content/venv", f"建立 Python {PY_VERSION} 環境")
    sh(f"uv pip install -q --python {VENV_PY} {PACKAGES}", "安裝套件")

out = sh([VENV_PY, "-c", CHECK], "檢查環境")
ver = json.loads(next(l for l in out.splitlines() if l.startswith("@@VER"))[5:])
display(HTML(table_html(["項目", "版本"], [[html.escape(k), f"<code>{v}</code>"] for k, v in ver.items()])))

In [ ]:
# @title 3️⃣ 上傳輸入檔 { display-mode: "form" }
# @markdown 按下方「選擇檔案」，選本機 `路徑/` 裡的檔案。
from google.colab import files

up = files.upload()
if not up:
    fail("沒有收到檔案", "請重新執行這一格。")
    raise StopRun("no input")
src = next(iter(up))
ok(f"已上傳：{src}（{len(up[src]) / 1024:.0f} KB）")

In [ ]:
# @title 4️⃣ 執行 { display-mode: "form" }
# @markdown 畫面只顯示進度表，完整訊息存在 `result/_run_log.txt`（會一起打包）。
script = r"""
import sys, time
items = ["項目1", "項目2", "項目3"]  # 換成實際工作
print("@@TOTAL", len(items), flush=True)
for i, it in enumerate(items, 1):
    time.sleep(1)  # 實際處理
    print("@@STEP", i, flush=True)
"""
open("job.py", "w", encoding="utf-8").write(script)
os.makedirs("result", exist_ok=True)

total, done = 0, 0


def board():
    bar = "█" * done + "░" * (total - done)
    return HTML(table_html(["工作", "進度"], [["執行", f"🔄 <code>{bar}</code> {done}/{total}"]]))


handle = display(board(), display_id=True)
t0 = time.time()
with open("result/_run_log.txt", "w", encoding="utf-8") as log_f:
    proc = subprocess.Popen([VENV_PY, "job.py", src], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log_f.write(line)
        parts = line.split()
        if parts and parts[0] == "@@TOTAL":
            total = int(parts[1])
        elif parts and parts[0] == "@@STEP":
            done = int(parts[1])
        else:
            continue
        handle.update(board())
    proc.wait()

if proc.returncode != 0:
    tail = open("result/_run_log.txt", encoding="utf-8").read().splitlines()[-15:]
    fail(f"執行失敗（代碼 {proc.returncode}）", logbox(chr(10).join(tail), "錯誤訊息最後 15 行"))
    raise StopRun("job failed")
ok(f"執行完成（{time.time() - t0:.0f} 秒）")

## 📦 結果

In [ ]:
# @title 5️⃣ 打包並下載結果 { display-mode: "form" }
import shutil

zip_path = shutil.make_archive("result", "zip", "result")
ok(f"已打包：result.zip（{os.path.getsize(zip_path) / 2**20:.1f} MB）",
   "瀏覽器會自動下載。請放到本機 <code>路徑/</code>。")
files.download(zip_path)